In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt

import sys
import os
from pathlib import Path

# move upward until we find repo root
repo_root = Path.cwd()

while not (repo_root / ".git").exists():
    repo_root = repo_root.parent

sys.path.append(str(repo_root))
from config import config
from DataPipeline.utils.bets_utils import generate_bets 

In [3]:
df_ml_history = pd.read_csv(config.ml_history_fp)
print(df_ml_history.shape)
df_ml_history = df_ml_history.drop_duplicates(subset=['fighter_red', 'fighter_blue', 'date'], keep='first')
print(df_ml_history.shape)

(48, 22)
(48, 22)


In [ ]:
df_features = pd.read_csv(r'C:\Users\jcmar\my_files\SportsBetting\Data\features_test_files\stats_odds_merged.csv')
for c in df_features.columns:
    print(c)

In [17]:
df_ml_history.tail()[['net_odds_open', 'fighter_red', 'fighter_blue', 'pred_name_open', 'pred_winner_open', 'pred_winner_close1_stack', 'pred_winner_close2_stack', 'winner_bool', 'date']]

,net_odds_open,fighter_red,fighter_blue,pred_name_open,pred_winner_open,pred_winner_close1_stack,pred_winner_close2_stack,winner_bool,date
43,300.0,vlasto cepo,gilbert urbina,gilbert urbina,0,NaN,NaN,0.0,2026-08-01 00:00:00
44,-200.0,jan blachowicz,navajo stirling,navajo stirling,0,NaN,NaN,0.0,2026-08-01 00:00:00
45,-1.0,aleksandar rakic,marcin tybura,marcin tybura,0,NaN,NaN,1.0,2026-08-01 00:00:00
46,-501.0,uros medic,daniel rodriguez,uros medic,1,1.0,1.0,1.0,2026-08-01 00:00:00
47,-1.0,dusko todorovic,robert valentin,robert valentin,0,0.0,0.0,3.0,2026-08-01 00:00:00


In [7]:
df_ml_history.columns

Index(['net_odds_open', 'net_odds_close1', 'net_odds_close2', 'winner_bool',
       'fighter_red', 'fighter_blue', 'pred_name_open', 'pred_name_close1',
       'pred_name_close2', 'pred_winner_open', 'pred_winner_close1_stack',
       'pred_winner_close2_stack', 'open_red', 'open_blue', 'close1_red',
       'close1_blue', 'close2_red', 'close2_blue', 'fstar_open',
       'fstar_close1', 'fstar_close2', 'date'],
      dtype='object')

In [5]:
df_parlay = pd.read_csv(config.parlay_history_fp)
df_parlay.columns

Index(['open_net_fstar', 'close1_net_fstar', 'close2_net_fstar', 'open_red',
       'open_blue', 'close1_red', 'close1_blue', 'close2_red', 'close2_blue',
       'open_net_odds', 'close1_net_odds', 'close2_net_odds', 'open_fstar',
       'close1_fstar', 'close2_fstar', 'choice_fighter_bool_open',
       'choice_fighter_bool_close1', 'choice_fighter_bool_close2',
       'winner_bool_open', 'winner_bool_close1', 'winner_bool_close2',
       'choice_fighter_name_open', 'choice_fighter_name_close1',
       'choice_fighter_name_close2', 'date'],
      dtype='object')

In [30]:
df_parlay.head()[['winner_bool_open', 'choice_fighter_name_open']]

,winner_bool_open,choice_fighter_name_open
0,1,nurullo aliev
1,0,steve erceg
2,3,aleksandar rakic
3,1,uros medic
4,1,nurullo aliev


In [8]:
def returns_by_date():
    
    df_ml = pd.read_csv(config.ml_history_fp)
    df_parlay = pd.read_csv(config.parlay_history_fp)

    types = ['open', 'close1', 'close2']
    ml_pct_returns = {type_:[] for type_ in types}
    parlay_pct_returns = {type_:[] for type_ in types}

    ml_pct_returns['date'] = []
    parlay_pct_returns['date'] = [] 

    for date, group in df_ml.groupby('date'): 
        
        parlay_group = df_parlay[df_parlay['date']==date]

        for type_ in types: 

            total_net_odds = np.sum(np.where(group[f'fstar_{type_}'] > 0, group[f'net_odds_{type_}'], 0))
            ml_pct_returns[type_].append(total_net_odds)

            valid_parlay = parlay_group[f'{type_}_fstar'].iloc[0] > 0 
            total_net_odds = parlay_group[f'{type_}_net_odds'].iloc[0] if valid_parlay else 0 
            parlay_pct_returns[type_].append(total_net_odds)
        
        ml_pct_returns['date'].append(date)
        parlay_pct_returns['date'].append(date)

    df_ml_pct = pd.DataFrame(ml_pct_returns)
    df_parlay_pct = pd.DataFrame(parlay_pct_returns)

    return df_ml_pct, df_parlay_pct


df_ml_pct, df_parlay_pct = returns_by_date()


In [9]:
df_ml_pct.head()

,open,close1,close2,date
0,-1173.0,-446.0,-914.0,2026-07-25
1,-1173.0,-446.0,-914.0,2026-07-25 00:00:00
2,-501.0,-435.0,-350.0,2026-08-01
3,-501.0,-435.0,-350.0,2026-08-01 00:00:00


In [29]:
df_stats_missing = pd.read_csv(config.non_merged_stats_fp)
print(df_stats_missing.shape[0])

df_stats_missing = df_stats_missing.drop_duplicates(subset=['fighter_red', 'fighter_blue', 'event_date'])
print(df_stats_missing.shape)

missing_odds_prev = pd.read_csv(config.non_merged_odds_fp)
print(missing_odds_prev.shape)

missing_odds_prev = missing_odds_prev.drop_duplicates(subset=['event_date', 'blue_fighter', 'red_fighter'])
print(missing_odds_prev.shape)

264
(238, 54)
(254, 11)
(254, 11)


In [27]:
missing_odds_prev.columns

Index(['blue_fighter', 'open_blue', 'close1_blue', 'close2_blue',
       'red_fighter', 'open_red', 'close1_red', 'close2_red', 'event_date',
       'og_blue_name', 'og_red_fighter'],
      dtype='object')

In [19]:
df_stats_missing.head()

,title_fight,event_name,event_date,event_location,fight_url,weight_class,method,round,fight_time,performance_bonus_winner,...,head_red,head_blue,body_red,body_blue,leg_red,leg_blue,distance_red,distance_blue,total_strikes_red,total_strikes_blue
0,0,UFC Fight Night: Evloev vs. Murphy,2026-03-21,"London, England, United Kingdom",http://ufcstats.com/fight-details/346e3897c2ea...,Featherweight,M-DEC,5,5:00,0,...,47 of 134,45 of 175,33 of 44,24 of 38,6 of 11,20 of 26,80 of 178,89 of 238,124 of 234,89 of 239
1,0,UFC Fight Night: Evloev vs. Murphy,2026-03-21,"London, England, United Kingdom",http://ufcstats.com/fight-details/fa534aaa5b99...,Featherweight,U-DEC,3,5:00,0,...,65 of 134,48 of 158,31 of 36,13 of 21,4 of 7,0 of 1,90 of 166,58 of 169,113 of 192,107 of 233
2,0,UFC Fight Night: Evloev vs. Murphy,2026-03-21,"London, England, United Kingdom",http://ufcstats.com/fight-details/5ee11b727e89...,Welterweight,U-DEC,3,5:00,0,...,17 of 38,4 of 50,4 of 8,4 of 6,6 of 9,4 of 8,27 of 55,10 of 60,33 of 61,25 of 78
3,0,UFC Fight Night: Evloev vs. Murphy,2026-03-21,"London, England, United Kingdom",http://ufcstats.com/fight-details/43e26b46cbb5...,Light Heavyweight,"KO/TKO, Punch",1,0:28,1,...,9 of 12,0 of 3,0 of 0,0 of 0,1 of 1,0 of 1,2 of 5,0 of 4,10 of 13,0 of 4
4,0,UFC Fight Night: Evloev vs. Murphy,2026-03-21,"London, England, United Kingdom",http://ufcstats.com/fight-details/562d916f5591...,Middleweight,U-DEC,3,5:00,0,...,8 of 31,18 of 52,0 of 0,6 of 12,2 of 3,30 of 35,9 of 33,51 of 92,40 of 94,78 of 125


In [4]:
df_ml_history = df_ml_history.drop_duplicates(subset=['fighter_red', 'fighter_blue'])
print(df_ml_history.shape)

(192, 20)


In [2]:
folder_path = r"C:\Users\jcmar\my_files\SportsBetting\Data\upcoming_events\event_features"

file_path = os.path.join(folder_path, "upcoming_odds_stats_2026-06-06.csv")
print(file_path)

upcoming_fp = fr'{file_path}'
upcoming_df = pd.read_csv(upcoming_fp)  

df_bets_all, df_parlay_all = generate_bets(upcoming_df)




C:\Users\jcmar\my_files\SportsBetting\Data\upcoming_events\event_features\upcoming_odds_stats_2026-06-06.csv
Constant-like columns: ['const']
Constant-like columns: ['const']
Constant-like columns: ['const']
0     0.282688
1     0.205283
2     0.219648
3     0.098323
4     0.014437
5     0.201905
6     0.007705
7     0.120093
8     0.102102
9          NaN
10    0.257567
11    0.000000
Name: fstar_scaled, dtype: float64
0     0.283114
1     0.205430
2     0.218685
3     0.065025
4     0.092367
5     0.219545
6     0.000000
7     0.119642
8     0.106465
9          NaN
10    0.260677
11    0.000000
Name: fstar_scaled, dtype: float64


In [4]:
upcoming_df[['open_red', 'open_blue', 'close2_red', 'close2_blue']] 

,open_red,open_blue,close2_red,close2_blue
0,-450.0,350.0,-520.0,400.0
1,-350.0,285.0,-225.0,195.0
2,-185.0,160.0,-275.0,250.0
3,220.0,-260.0,-250.0,220.0
4,400.0,-550.0,550.0,-549.0
5,-250.0,210.0,-155.0,135.0
6,-110.0,-110.0,-132.0,115.0
7,-110.0,-110.0,100.0,-120.0
8,330.0,-400.0,165.0,-175.0
9,-900.0,600.0,-350.0,310.0
